# Train a miniGPT language model with JAX on a single TPUAdapted from the official [JAX miniGPT tutorial](https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html)for the **single-chip TPU** that Colab provides.**What changed and why.** The original builds its headline feature on a `(4, 2)` device mesh —4-way data parallel by 2-way tensor parallel — which needs 8 devices and targets Kaggle'sTPU v5e-8. Colab offers **v5e-1 and v6e-1, one chip**, so none of that runs here. Rather thanleave dead code behind a `device_count() == 1` branch, the sharding is removed entirely and thenotebook focuses on what a single chip can actually teach. Multi-device parallelism belongs inits own notebook on 8-chip hardware.**You will:**1. Build miniGPT with Flax NNX2. Train it on TinyStories3. Generate text4. Look at *why* the first step is slow and how batch size moves throughputMeasured on Colab **TPU v5e-1**, 2026-11-01.

## 1. Get the data firstTinyStories is **1.8 GB** and takes a few minutes. Start it now — the cell blocks, so this is agood moment to talk through the Transformer itself.

In [ ]:
!wget -q --show-progress \  "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt?download=true" \  -O TinyStories-train.txt!ls -lh TinyStories-train.txt

## 2. Pin the environmentColab preinstalls `jax 0.11.2` with `flax 0.11.2`, and **that pair cannot import `flax.nnx`** —the old Flax reaches for `jax._src.core.mutable_array`, which this JAX no longer has. So a newerFlax is required, not optional.Every version is pinned. A notebook that floats on whatever the runtime happens to ship stopsbeing reproducible the moment the runtime moves.

In [ ]:
!pip install -q \    "jax[tpu]==0.12.0.1" \    "flax==0.12.12" \    "optax==0.2.8" \    "grain==0.2.18" \    "tiktoken==0.14.0"

### Restart the runtime now**Runtime > Restart session**, then continue from the next cell.`pip` wrote new files to disk, but Python keeps the already-imported modules. Skip the restartand you get the old JAX with the new Flax — which fails with the same `mutable_array``ImportError` described above. The next cell asserts the versions so this cannot pass silently.

In [ ]:
import jax, flax, optaxEXPECTED = {"jax": "0.12.0.1", "flax": "0.12.12", "optax": "0.2.8"}for mod in (jax, flax, optax):    got, want = mod.__version__, EXPECTED[mod.__name__]    assert got == want, (        f"{mod.__name__} is {got}, expected {want}. "        "Restart the runtime (Runtime > Restart session) and run this cell again."    )print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__)print(jax.devices())assert any(d.platform == "tpu" for d in jax.devices()), \    "No TPU. Runtime > Change runtime type > TPU."

## 3. Imports and hyperparametersThese match the original tutorial. `batch_size = 144` was chosen there for 8 devices; it wasre-measured here and still works on one chip, so it stays (see the last section).

In [ ]:
import timeimport jax.numpy as jnpimport optaximport pandas as pdimport tiktokenimport grain.python as pygrainfrom flax import nnxvocab_size = 50257          # GPT-2 tokenizermaxlen = 256embed_dim = 256num_heads = 8feed_forward_dim = 256num_transformer_blocks = 8batch_size = 144num_epochs = 1max_steps = 500             # cap the run; a full epoch is ~30 min on v5e-1

## 4. The modelNote `nnx.List` rather than a plain Python list. Assigning a bare list makes Flax treat it as a*static* attribute and the model will not train:```ValueError: Found data on value of type '<class 'list'>' assigned to static attribute 'blocks'```

In [ ]:
def causal_attention_mask(seq_len):    return jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))class TransformerBlock(nnx.Module):    def __init__(self, embed_dim, num_heads, ff_dim, *, rngs, rate=0.1):        self.attn = nnx.MultiHeadAttention(            num_heads=num_heads, in_features=embed_dim, qkv_features=embed_dim,            decode=False, rngs=rngs,        )        self.norm1 = nnx.LayerNorm(embed_dim, epsilon=1e-6, rngs=rngs)        self.norm2 = nnx.LayerNorm(embed_dim, epsilon=1e-6, rngs=rngs)        self.dense1 = nnx.Linear(embed_dim, ff_dim, rngs=rngs)        self.dense2 = nnx.Linear(ff_dim, embed_dim, rngs=rngs)        self.drop1 = nnx.Dropout(rate=rate, rngs=rngs)        self.drop2 = nnx.Dropout(rate=rate, rngs=rngs)    def __call__(self, inputs, training=False):        mask = causal_attention_mask(inputs.shape[1])        attn = self.attn(inputs, mask=mask, decode=False)        attn = self.drop1(attn, deterministic=not training)        x = self.norm1(inputs + attn)        ff = self.dense2(nnx.relu(self.dense1(x)))        ff = self.drop2(ff, deterministic=not training)        return self.norm2(x + ff)class TokenAndPositionEmbedding(nnx.Module):    def __init__(self, maxlen, vocab_size, embed_dim, *, rngs):        self.token_emb = nnx.Embed(num_embeddings=vocab_size, features=embed_dim, rngs=rngs)        self.pos_emb = nnx.Embed(num_embeddings=maxlen, features=embed_dim, rngs=rngs)    def __call__(self, x):        return self.token_emb(x) + self.pos_emb(jnp.arange(x.shape[1]))class MiniGPT(nnx.Module):    def __init__(self, maxlen, vocab_size, embed_dim, num_heads, ff_dim, num_blocks, *, rngs):        self.embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim, rngs=rngs)        # nnx.List, not [] — see the note above.        self.blocks = nnx.List(            [TransformerBlock(embed_dim, num_heads, ff_dim, rngs=rngs) for _ in range(num_blocks)]        )        self.output_layer = nnx.Linear(embed_dim, vocab_size, rngs=rngs)    def __call__(self, inputs, training=False):        x = self.embedding(inputs)        for block in self.blocks:            x = block(x, training=training)        return self.output_layer(x)model = MiniGPT(maxlen, vocab_size, embed_dim, num_heads, feed_forward_dim,                num_transformer_blocks, rngs=nnx.Rngs(0))n_params = sum(x.size for x in jax.tree.leaves(nnx.state(model, nnx.Param)))print(f"parameters: {n_params:,}")

## 5. Data pipeline

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")class TextDataset:    def __init__(self, data, maxlen):        self.data = data        self.maxlen = maxlen    def __len__(self):        return len(self.data)    def __getitem__(self, idx):        ids = tokenizer.encode(self.data[idx], allowed_special={"<|endoftext|>"})[: self.maxlen]        return jnp.array(ids + [0] * (self.maxlen - len(ids)), dtype=jnp.int32)def load_and_preprocess_data(file_path, batch_size, maxlen):    with open(file_path, "r") as f:        text = f.read()    stories = [s + "<|endoftext|>" for s in text.split("<|endoftext|>") if s.strip()]    dataset = TextDataset(pd.Series(stories).dropna().tolist(), maxlen)    sampler = pygrain.IndexSampler(        len(dataset), shuffle=False, seed=42,        shard_options=pygrain.NoSharding(), num_epochs=num_epochs,    )    return pygrain.DataLoader(        data_source=dataset, sampler=sampler,        operations=[pygrain.Batch(batch_size=batch_size, drop_remainder=True)],    )text_dl = load_and_preprocess_data("TinyStories-train.txt", batch_size, maxlen)

## 6. Loss and training step`nnx.jit` compiles the whole step. The first call pays for tracing and XLA compilation; latercalls hit the cache. Section 8 measures that gap.

In [ ]:
def loss_fn(model, batch):    logits = model(batch[:, :-1], training=True)    return optax.softmax_cross_entropy_with_integer_labels(logits, batch[:, 1:]).mean()@nnx.jitdef train_step(model, optimizer, batch):    loss, grads = nnx.value_and_grad(loss_fn)(model, batch)    optimizer.update(model, grads)    return lossoptimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

## 7. TrainCapped at `max_steps` so the cell finishes in a couple of minutes. A full epoch over TinyStoriesis roughly 30 minutes at 128.7 ms per step.

In [ ]:
losses, start = [], time.perf_counter()for step, batch in enumerate(text_dl):    if step >= max_steps:        break    loss = train_step(model, optimizer, batch)    if step % 50 == 0:        loss = float(loss)        losses.append((step, loss))        print(f"step {step:4d}  loss {loss:.4f}  elapsed {time.perf_counter() - start:6.1f}s")print(f"done in {time.perf_counter() - start:.1f}s")

## 8. Why the first step is slow

In [ ]:
batch = next(iter(text_dl))m2 = MiniGPT(maxlen, vocab_size, embed_dim, num_heads, feed_forward_dim,             num_transformer_blocks, rngs=nnx.Rngs(1))o2 = nnx.Optimizer(m2, optax.adam(1e-3), wrt=nnx.Param)t0 = time.perf_counter()jax.block_until_ready(train_step(m2, o2, batch))first = time.perf_counter() - t0times = []for _ in range(5):    t0 = time.perf_counter()    jax.block_until_ready(train_step(m2, o2, batch))    times.append(time.perf_counter() - t0)steady = sorted(times)[len(times) // 2]print(f"first step : {first:6.2f} s   (tracing + XLA compilation)")print(f"steady step: {steady * 1000:6.1f} ms")print(f"ratio      : {first / steady:6.0f}x")print(f"throughput : {batch.shape[0] * maxlen / steady:,.0f} tokens/s")

**`jax.block_until_ready` is not optional.** JAX dispatches asynchronously, so timing without itmeasures how fast Python queued the work, not how fast the TPU did it.

## 9. Batch size and throughputMeasured on v5e-1 with this model, 2026-11-01:| batch | first step | steady | tokens/s ||---:|---:|---:|---:|| 16 | 33.99 s | 23.2 ms | 176,824 || 32 | 44.02 s | 34.7 ms | 235,907 || 64 | 44.07 s | 65.1 ms | 251,626 || **144** | 46.70 s | 128.7 ms | **286,381** |Nine times the batch buys 1.6x the throughput. At small batch the chip spends much of each stepwaiting between dispatched programs rather than computing, which is why `batch_size = 144` — thevalue the original tutorial picked for eight devices — is still a reasonable default on one.It also fits: no out-of-memory at 144 on v5e-1.

## 10. Generate text

In [ ]:
def generate(model, prompt, max_tokens=64, top_k=10, seed=0):    key = jax.random.key(seed)    tokens = tokenizer.encode(prompt, allowed_special={"<|endoftext|>"})    for _ in range(max_tokens):        window = tokens[-maxlen:]        padded = jnp.array(window + [0] * (maxlen - len(window)), dtype=jnp.int32)[None, :]        logits = model(padded, training=False)[0, len(window) - 1]        top_logits, top_ids = jax.lax.top_k(logits, top_k)        key, sub = jax.random.split(key)        choice = jax.random.categorical(sub, top_logits)        tokens.append(int(top_ids[choice]))    return tokenizer.decode(tokens)print(generate(model, "Once upon a time", max_tokens=48))

## What this notebook does not cover**Multi-device parallelism.** The original tutorial's `(4, 2)` mesh needs 8 chips. Colab givesone, so data and tensor parallelism are out of scope here and deserve their own notebook onKaggle TPU v5e-8 or a Cloud TPU slice.---Code and measurements: [`tsho-lab/jax-tpu-minigpt`](https://github.com/tsho/tsho-lab/tree/main/jax-tpu-minigpt) · Apache-2.0